# Ghost in the Aether — Housekeeping / Reset

Run this **before each presentation** to clear the previous run's audience votes (and,
optionally, the streamed Security/Comms demo data) so the dashboards start clean.

What it does:
- Auto-resolves the **AetherEH** Eventhouse cluster URI from the current workspace (zero config).
- Shows row counts **before**.
- Runs `.clear table <name> data` — this deletes all rows but **keeps the table schema**,
  so the Eventstream keeps writing and the dashboard tiles keep working.
- Shows row counts **after** to confirm the reset.

**Defaults:** clears the `Votes` table only. Set `RESET_DEMO_DATA = True` in the config cell to
also clear the streamed `SecurityLogs` and `Communications` tables (useful before re-running the
Event Simulator). The reference tables `VictimCalendar` and `SupplierRecords` are left untouched.

> Tip: clear right at showtime. Any votes still buffered in the Event Hub / Eventstream will land
> *after* the clear, so give in-flight submissions a moment to drain (or briefly pause the
> Eventstream source) before running.

In [ ]:
%pip install azure-kusto-data==4.5.1

In [ ]:
# --- Configuration ---
DATABASE = "AetherEH"          # KQL database name
EVENTHOUSE_NAME = "AetherEH"   # Eventhouse item display name

# Which tables to reset.
VOTE_TABLES = ["Votes"]                             # always cleared
DEMO_TABLES = ["SecurityLogs", "Communications"]    # streamed demo data
# Reference tables (VictimCalendar, SupplierRecords) are populated once and NOT reset.

RESET_DEMO_DATA = False   # set True to also clear SecurityLogs + Communications

# Optional manual override. Leave blank to auto-resolve from the workspace.
CLUSTER_URI = ""

print("Reset target(s):", VOTE_TABLES + (DEMO_TABLES if RESET_DEMO_DATA else []))

In [ ]:
# --- Resolve the Eventhouse cluster URI from the current workspace ---
import requests
import notebookutils

ctx = notebookutils.runtime.context
WORKSPACE_ID = ctx.get("currentWorkspaceId") or ctx.get("workspaceId")
print("Workspace:", WORKSPACE_ID)

if not CLUSTER_URI:
    fab_token = notebookutils.credentials.getToken("pbi")
    resp = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/eventhouses",
        headers={"Authorization": f"Bearer {fab_token}"},
    )
    resp.raise_for_status()
    eventhouses = resp.json().get("value", [])
    match = next((e for e in eventhouses if e.get("displayName") == EVENTHOUSE_NAME), None)
    if match is None:
        raise RuntimeError(f"Eventhouse '{EVENTHOUSE_NAME}' not found in workspace {WORKSPACE_ID}")
    CLUSTER_URI = match["properties"]["queryServiceUri"]

print("Cluster:", CLUSTER_URI)

In [ ]:
# --- Build the Kusto client (uses the notebook identity's AAD token) ---
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder

kcsb = KustoConnectionStringBuilder.with_aad_application_token_authentication(
    CLUSTER_URI, notebookutils.credentials.getToken(CLUSTER_URI)
)
client = KustoClient(kcsb)

def row_count(table):
    try:
        result = client.execute(DATABASE, f"{table} | count")
        return result.primary_results[0][0]["Count"]
    except Exception as exc:
        return f"(error: {exc})"

def show_counts(label, tables):
    print(f"--- {label} ---")
    for t in tables:
        print(f"  {t}: {row_count(t)} rows")

print("Kusto client ready")

In [ ]:
tables_to_clear = list(VOTE_TABLES) + (list(DEMO_TABLES) if RESET_DEMO_DATA else [])
show_counts("BEFORE reset", tables_to_clear)

In [ ]:
# --- Clear data (schema preserved) ---
for t in tables_to_clear:
    client.execute_mgmt(DATABASE, f".clear table {t} data")
    print(f"Cleared {t}")

print("\nReset complete.")

In [ ]:
show_counts("AFTER reset", tables_to_clear)